In [1]:
import pandas as pd
import geopandas as gpd

In [4]:
pip install census

Note: you may need to restart the kernel to use updated packages.


In [46]:
import os
import zipfile
import requests
import geopandas as gpd
import pandas as pd
from census import Census
from pathlib import Path

In [48]:


# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------
# Get a free API key at: https://api.census.gov/data/key_signup.html
CENSUS_API_KEY = "db87451d2114e8ee87f1fc8d516119219cae4699"

# ACS 5-Year Table B08301: Means of Transportation to Work
# Variables mapping:
# B08301_001E: Total workers 16+
# B08301_003E: Car, truck, or van -- drove alone
# B08301_004E: Car, truck, or van -- Carpooled
# B08301_010E: Public transportation (excluding Taxicab)
# B08301_018E: Bicycle
# B08301_019E: Walked
# B08301_021E: Worked from home
VARIABLES = [
    "NAME",
    "B08301_001E", "B08301_003E", "B08301_004E",
    "B08301_010E", "B08301_016E", "B08301_017E", "B08301_018E", "B08301_019E", "B08301_020E", "B08301_021E"
]

RENAME_MAP = {
    "B08301_001E": "Total Workers",
    "B08301_003E": "Drove Alone",
    "B08301_004E": "Carpooled",
    "B08301_010E": "Public Transit",
    "B08301_016E": "Taxicab",
    "B08301_017E": "Motorcycle",
    "B08301_018E": "Bicycle",
    "B08301_019E": "Walked",
    "B08301_020E": "Other",
    "B08301_021E": "Work From Home"
}

YEAR = 2024  # Latest available ACS 5-Year dataset
PMTILES_OUT = "/Users/root1/apps/maps/austin/app/templates/static/tiles/transportation/census_tracts_commute.pmtiles"
c = Census(CENSUS_API_KEY)


# Set your target directory
folder_path = Path("/Users/root1/apps/maps/shapefiles/2024/")

# Filter only items that are directories
subdirs = [f.name for f in folder_path.iterdir() if f.is_dir()]
subdirs.sort()
subdirs.remove("60")
subdirs.remove("66")
subdirs.remove("69")
subdirs.remove("72")
subdirs.remove("78")
for STATE_FIPS in subdirs:
    print(f"working on {STATE_FIPS}")
    GEOJSON_OUT=f"transportation/shapes_{STATE_FIPS}.geojson"
    # Query census tracts for the specified state
    acs_raw = c.acs5.state_county_tract(
        fields=VARIABLES,
        state_fips=STATE_FIPS,
        county_fips=Census.ALL,
        tract=Census.ALL,
        year=YEAR
    )
    
    df_acs = pd.DataFrame(acs_raw)
    
    # Generate a 11-digit GEOID matching TIGER/Line standard (State + County + Tract)
    df_acs["GEOID"] = df_acs["state"] + df_acs["county"] + df_acs["tract"]
    
    # Rename and convert data columns to numeric
    df_acs.rename(columns=RENAME_MAP, inplace=True)
    num_cols = list(RENAME_MAP.values())
    df_acs[num_cols] = df_acs[num_cols].apply(pd.to_numeric, errors="coerce")
    
    # Calculate percentage metrics
    df_acs["Percent Drove Alone"] = (df_acs["Drove Alone"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Carpooled"] = (df_acs["Carpooled"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Public Transit"] = (df_acs["Public Transit"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Taxicab"] = (df_acs["Taxicab"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Motorcycle"] = (df_acs["Motorcycle"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Bicycle"] = (df_acs["Bicycle"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Walked"] = (df_acs["Walked"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Other"] = (df_acs["Other"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Percent Working From Home"] = (df_acs["Work From Home"] / df_acs["Total Workers"] * 100).round(2).fillna(0)
    df_acs["Car Free Commuters"] = ((df_acs["Bicycle"] + df_acs['Walked'] + df_acs['Public Transit'] + df_acs['Other']) / df_acs['Total Workers'] * 100).round(2).fillna(0)
    
    
    # Keep key fields
    df_acs = df_acs[["GEOID"] + ["Total Workers", "Percent Drove Alone", "Percent Carpooled", "Percent Public Transit", "Percent Taxicab", "Percent Motorcycle", "Percent Bicycle","Percent Walked","Percent Other", "Percent Working From Home", "Car Free Commuters"]]
    
    print("Joining shapefiles with ACS attributes...")
    shp_path = f"../shapefiles/2024/{STATE_FIPS}/tl_2024_{STATE_FIPS}_tract.shp"
    gdf_tracts = gpd.read_file(shp_path)
    
    # Merge on 11-character GEOID string
    gdf_merged = gdf_tracts[["GEOID", "geometry"]].merge(df_acs, on="GEOID", how="inner")
    
    # Reproject to WGS84 (EPSG:4326) as required by Tippecanoe
    gdf_merged = gdf_merged.to_crs(epsg=4326)
    gdf_merged = gdf_merged[gdf_merged['Total Workers']>0]
    print(f"Exporting merged dataset to {GEOJSON_OUT}...")
    gdf_merged.to_file(GEOJSON_OUT, driver="GeoJSON")

# ------------------------------------------------------------------
# 4. RENDER TO PMTILES USING TIPPECANOE
# ------------------------------------------------------------------
print(f"Generating PMTiles archive: {PMTILES_OUT}...")

# Tippecanoe natively writes to `.pmtiles` files directly via -o
tippecanoe_cmd = f"""
tippecanoe \
-zg -o /Users/root1/apps/maps/austin/app/templates/static/tiles/transportation/nationwide.pmtiles \
-l counties \
--coalesce-densest-as-needed \
--extend-zooms-if-still-dropping \
transportation/shapes_01.geojson \
transportation/shapes_02.geojson \
transportation/shapes_04.geojson \
transportation/shapes_05.geojson \
transportation/shapes_06.geojson \
transportation/shapes_08.geojson \
transportation/shapes_09.geojson \
transportation/shapes_10.geojson \
transportation/shapes_11.geojson \
transportation/shapes_12.geojson \
transportation/shapes_13.geojson \
transportation/shapes_15.geojson \
transportation/shapes_16.geojson \
transportation/shapes_17.geojson \
transportation/shapes_18.geojson \
transportation/shapes_19.geojson \
transportation/shapes_20.geojson \
transportation/shapes_21.geojson \
transportation/shapes_22.geojson \
transportation/shapes_23.geojson \
transportation/shapes_24.geojson \
transportation/shapes_25.geojson \
transportation/shapes_26.geojson \
"""
os.system(tippecanoe_cmd)


tippecanoe_cmd = f"""
tippecanoe \
-zg -o /Users/root1/apps/maps/austin/app/templates/static/tiles/transportation/nationwide2.pmtiles \
-l counties \
--coalesce-densest-as-needed \
--extend-zooms-if-still-dropping \
transportation/shapes_27.geojson \
transportation/shapes_28.geojson \
transportation/shapes_29.geojson \
transportation/shapes_30.geojson \
transportation/shapes_31.geojson \
transportation/shapes_32.geojson \
transportation/shapes_33.geojson \
transportation/shapes_34.geojson \
transportation/shapes_35.geojson \
transportation/shapes_36.geojson \
transportation/shapes_37.geojson \
transportation/shapes_38.geojson \
transportation/shapes_39.geojson \
transportation/shapes_40.geojson \
transportation/shapes_41.geojson \
transportation/shapes_42.geojson \
transportation/shapes_44.geojson \
transportation/shapes_45.geojson \
transportation/shapes_46.geojson \
transportation/shapes_47.geojson \
transportation/shapes_48.geojson \
transportation/shapes_49.geojson \
transportation/shapes_50.geojson \
transportation/shapes_51.geojson \
transportation/shapes_53.geojson \
transportation/shapes_54.geojson \
transportation/shapes_55.geojson \
transportation/shapes_56.geojson \
"""
os.system(tippecanoe_cmd)
print("PMTiles file generation complete!")

working on 01


KeyboardInterrupt: 

In [49]:

# Tippecanoe natively writes to `.pmtiles` files directly via -o
tippecanoe_cmd = f"""
tippecanoe \
-zg -o /Users/root1/apps/maps/austin/app/templates/static/tiles/transportation/nationwide.pmtiles \
-l counties \
--coalesce-densest-as-needed \
--extend-zooms-if-still-dropping \
transportation/shapes_01.geojson \
transportation/shapes_02.geojson \
transportation/shapes_04.geojson \
transportation/shapes_05.geojson \
transportation/shapes_06.geojson \
transportation/shapes_08.geojson \
transportation/shapes_09.geojson \
transportation/shapes_10.geojson \
transportation/shapes_11.geojson \
transportation/shapes_12.geojson \
transportation/shapes_13.geojson \
transportation/shapes_15.geojson \
transportation/shapes_16.geojson \
transportation/shapes_17.geojson \
transportation/shapes_18.geojson \
transportation/shapes_19.geojson \
transportation/shapes_20.geojson \
transportation/shapes_21.geojson \
transportation/shapes_22.geojson \
transportation/shapes_23.geojson \
transportation/shapes_24.geojson \
transportation/shapes_25.geojson \
transportation/shapes_26.geojson \
"""
os.system(tippecanoe_cmd)


tippecanoe_cmd = f"""
tippecanoe \
-zg -o /Users/root1/apps/maps/austin/app/templates/static/tiles/transportation/nationwide2.pmtiles \
-l counties \
--coalesce-densest-as-needed \
--extend-zooms-if-still-dropping \
transportation/shapes_27.geojson \
transportation/shapes_28.geojson \
transportation/shapes_29.geojson \
transportation/shapes_30.geojson \
transportation/shapes_31.geojson \
transportation/shapes_32.geojson \
transportation/shapes_33.geojson \
transportation/shapes_34.geojson \
transportation/shapes_35.geojson \
transportation/shapes_36.geojson \
transportation/shapes_37.geojson \
transportation/shapes_38.geojson \
transportation/shapes_39.geojson \
transportation/shapes_40.geojson \
transportation/shapes_41.geojson \
transportation/shapes_42.geojson \
transportation/shapes_44.geojson \
transportation/shapes_45.geojson \
transportation/shapes_46.geojson \
transportation/shapes_47.geojson \
transportation/shapes_48.geojson \
transportation/shapes_49.geojson \
transportation/shapes_50.geojson \
transportation/shapes_51.geojson \
transportation/shapes_53.geojson \
transportation/shapes_54.geojson \
transportation/shapes_55.geojson \
transportation/shapes_56.geojson \
"""
os.system(tippecanoe_cmd)
print("PMTiles file generation complete!")

40412 features, 95273612 bytes of geometry and attributes, 615231 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z5 for features typically 11459 feet (3493 meters) apart, and at least 2143 feet (653 meters) apart
Choosing a maxzoom of -z11 for resolution of about 233 feet (71 meters) within features
tile 3/2/3 size is 544256 with detail 12, >500000    
Going to try keeping the sparsest 73.49% of the features to make it fit
tile 4/4/6 size is 681391 with detail 12, >500000    
Going to try keeping the sparsest 58.70% of the features to make it fit
  99.9%  11/336/800   
43047 features, 125455747 bytes of geometry and attributes, 650290 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z5 for features typically 13107 feet (3995 meters) apart, and at least 2285 feet (697 meters) apart
Choosing a maxzoom of -z11 for resolution of about 219 feet (66 meters) within features
tile 1/0/0 size is 526975 with detail 12, >500000 

PMTiles file generation complete!


In [32]:
acs_raw

[]

In [34]:
num_cols

['workers_total',
 'drove_alone',
 'carpooled',
 'public_transit',
 'taxicab',
 'motorcycle',
 'bicycle',
 'walked',
 'other',
 'worked_home']

In [44]:
gdf_merged[gdf_merged['Total Workers']>0]

,GEOID,geometry,Total Workers,Percent Drove Alone,Percent Carpooled,Percent Public Transit,Percent Taxicab,Percent Motorcycle,Percent Bicycle,Percent Walked,Percent Other,Percent Working From Home
0,56031959402,"POLYGON ((-105.27897 41.742, -105.27181 41.742...",688.0,75.00,10.17,0.00,0.0,0.00,0.00,1.74,0.00,13.08
1,56031959401,"POLYGON ((-105.28145 42.12274, -105.28144 42.1...",1952.0,82.38,7.27,0.00,0.0,0.00,0.00,4.25,1.28,4.82
2,56011950200,"POLYGON ((-105.08615 44.52771, -105.0861 44.52...",2125.0,74.59,10.54,0.00,0.0,0.00,0.89,5.65,0.00,8.33
3,56011950300,"POLYGON ((-104.69674 44.57976, -104.69673 44.5...",1458.0,69.20,12.07,0.00,0.0,0.55,0.00,2.74,0.00,15.43
4,56037970905,"POLYGON ((-109.31426 41.5945, -109.31381 41.59...",3113.0,80.69,11.92,1.25,0.0,0.00,0.48,0.64,1.77,3.24
...,...,...,...,...,...,...,...,...,...,...,...,...
155,56021000200,"POLYGON ((-104.85109 41.11787, -104.85084 41.1...",2762.0,74.26,22.05,0.18,0.0,0.00,0.00,0.72,0.04,2.75
156,56009956500,"POLYGON ((-105.4736 42.72471, -105.4731 42.724...",1457.0,80.44,6.18,2.33,0.0,0.00,0.00,3.02,0.69,7.34
157,56009956400,"POLYGON ((-105.42173 42.80178, -105.42172 42.8...",2335.0,74.00,9.98,3.90,0.0,0.00,0.00,5.74,1.33,5.05
158,56009956600,"POLYGON ((-106.07824 43.47968, -106.0782 43.48...",1619.0,67.26,15.50,2.47,0.0,0.00,0.00,2.16,0.00,12.60


In [42]:

    df_acs["Percent Alternative Commuters"] = ((df_acs["Bicycle"] + df_acs['Walked'] + df_acs['Public Transit'] + df_acs['Other']) / df_acs['Total Workers'] * 100).round(2).fillna(0)

KeyError: 'Bicycle'